<a href="https://colab.research.google.com/github/ericyoc/QDFIR_Hybrid_Confirmation_CERT_poc/blob/main/QDFIR_Hybrid_Confirmation_CERT_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess, glob, hashlib, time, json, math, shutil, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    sh("pip -q install pennylane pyarrow")
    sh("apt-get -qq install -y lbzip2 > /dev/null")
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/QDFIR_Empirical"
    SCRATCH = "/content/cert_scratch"
else:  # local test mode
    ROOT = os.environ.get("QDFIR_ROOT", "./QDFIR_Empirical")
    SCRATCH = os.path.join(ROOT, "_scratch")

DATA, RAW, RES = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/results"
OUT = f"{RES}/confirmation"
for d in (DATA, RAW, RES, SCRATCH, OUT): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, torch, torch.nn as nn, requests
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")
if IN_COLAB and DEV.type != "cuda":
    print("WARNING: no GPU attached. Runtime -> Change runtime type -> GPU.")

QUICK = os.environ.get("QDFIR_QUICK") == "1"   # local smoke test only
CFG = dict(release="r4.2", n_qubits=4, q_layers=3, full_width=32,
           epochs=3 if QUICK else 40, tune_epochs=2 if QUICK else 20, patience=8, batch=4096,
           lr_grid=[3e-4, 1e-3, 3e-3, 1e-2, 3e-2], tune_seed=0, seeds=[1, 2, 3, 4, 5],
           widths=[4, 8, 16, 32], insider_fracs=[0.10, 0.25, 0.50], fpr_target=0.01,
           collapse_pr_auc=0.05, work_start=8, work_end=18, baseline_days=30)

# ------------------------------------------------------------------ 1. data
FILES = {  # KiltHub/figshare file ids and MD5s for article 12841247
    "r4.2.tar.bz2":    (24856766, "cf64caa378acb77cd0c608a5576d998c"),
    "answers.tar.bz2": (24857828, "c0c96eaa06915e417504f932756f58d8"),
}
def md5(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def download(name):
    fid, want = FILES[name]; dst = f"{RAW}/{name}"
    if os.path.exists(dst) and os.path.exists(dst + ".ok"): return dst
    url = f"https://ndownloader.figshare.com/files/{fid}"
    tmp = f"{SCRATCH}/{name}"
    print(f"Downloading {name} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status(); tot = int(r.headers.get("content-length", 0)); got = 0; t0 = time.time()
        with open(tmp, "wb") as f:
            for b in r.iter_content(1 << 23):
                f.write(b); got += len(b)
                if tot and got % (1 << 29) < (1 << 23):
                    print(f"  {got/1e9:.2f}/{tot/1e9:.2f} GB  {got/1e6/(time.time()-t0+1e-9):.0f} MB/s")
    assert md5(tmp) == want, f"MD5 mismatch for {name}"
    shutil.copy(tmp, dst); open(dst + ".ok", "w").write(want)
    return tmp  # local copy is faster to decompress

LOGS = ["logon", "device", "file"]
CSV_DIR = f"{DATA}/{CFG['release']}"
os.makedirs(CSV_DIR, exist_ok=True)
FEAT = f"{DATA}/{CFG['release']}_userday_features.parquet"

if not os.path.exists(FEAT):
    if not all(os.path.exists(f"{CSV_DIR}/{l}.csv") for l in LOGS):
        src = download("r4.2.tar.bz2")
        if not src.startswith(SCRATCH):
            shutil.copy(src, f"{SCRATCH}/r4.2.tar.bz2"); src = f"{SCRATCH}/r4.2.tar.bz2"
        print("Extracting logon/device/file logs (lbzip2, parallel) ...")
        dec = "-I lbzip2" if shutil.which("lbzip2") else "-j"
        pats = " ".join(f"'*{l}.csv'" for l in LOGS)
        sh(f"tar {dec} -xf {src} -C {SCRATCH} --wildcards {pats}")
        for l in LOGS:
            hit = [p for p in glob.glob(f"{SCRATCH}/**/{l}.csv", recursive=True)]
            assert hit, f"{l}.csv not found in archive"
            shutil.copy(hit[0], f"{CSV_DIR}/{l}.csv")
    if not glob.glob(f"{DATA}/answers/**/{CFG['release']}-*.csv", recursive=True):
        src = download("answers.tar.bz2")
        sh(f"tar -xjf {src} -C {DATA}")

    # ------------------------------------------------------- 2. features
    print("Building user-day features ...")
    FMT = "%m/%d/%Y %H:%M:%S"
    def load(name, cols):
        df = pd.read_csv(f"{CSV_DIR}/{name}.csv", usecols=cols)
        df["date"] = pd.to_datetime(df["date"], format=FMT)
        df["day"] = df["date"].dt.normalize(); df["hour"] = df["date"].dt.hour
        df["after"] = ((df["hour"] < CFG["work_start"]) | (df["hour"] >= CFG["work_end"])
                       | (df["date"].dt.dayofweek >= 5)).astype(np.int8)
        return df
    lg = load("logon", ["date", "user", "pc", "activity"])
    dv = load("device", ["date", "user", "pc", "activity"])
    fl = load("file", ["date", "user", "pc", "filename"])
    k = ["user", "day"]
    on = lg[lg.activity == "Logon"]; off = lg[lg.activity == "Logoff"]
    F = pd.concat([
        on.groupby(k).size().rename("n_logon"),
        off.groupby(k).size().rename("n_logoff"),
        on.groupby(k)["after"].sum().rename("n_logon_after"),
        lg.groupby(k)["pc"].nunique().rename("n_pc"),
        on.groupby(k)["hour"].min().rename("first_hour"),
        off.groupby(k)["hour"].max().rename("last_hour"),
        dv[dv.activity == "Connect"].groupby(k).size().rename("n_usb"),
        dv[dv.activity == "Connect"].groupby(k)["after"].sum().rename("n_usb_after"),
        dv.groupby(k)["pc"].nunique().rename("n_usb_pc"),
        fl.groupby(k).size().rename("n_file"),
        fl.groupby(k)["after"].sum().rename("n_file_after"),
    ], axis=1)
    ext = fl["filename"].str.extract(r"\.(\w+)$")[0].str.lower()
    for e in ["exe", "zip", "doc", "pdf", "txt", "jpg"]:
        F[f"n_file_{e}"] = fl[ext == e].groupby(k).size()
    F = F.reset_index()
    F["first_hour"] = F["first_hour"].fillna(F["first_hour"].median())
    F["last_hour"] = F["last_hour"].fillna(F["last_hour"].median())
    F = F.fillna(0)
    F["weekend"] = (F["day"].dt.dayofweek >= 5).astype(np.int8)
    # causal behavioural baseline: deviation from the user's own previous N days
    F = F.sort_values(k).reset_index(drop=True)
    base_cols = [c for c in F.columns if c.startswith("n_") or c in ("first_hour", "last_hour")]
    prev = (F.groupby("user")[base_cols]
              .transform(lambda s: s.shift(1).rolling(CFG["baseline_days"], min_periods=1).mean()))
    for c in base_cols: F[f"dev_{c}"] = F[c] - prev[c].fillna(F[c])

    # labels: a user-day is malicious if the answer key lists any malicious
    # event (logon/device/file/email/http) for that user on that day
    rows = []
    for p in glob.glob(f"{DATA}/answers/**/{CFG['release']}-*.csv", recursive=True):
        with open(p, errors="ignore") as f:
            for line in f:
                parts = line.strip().split(",", 4)
                if len(parts) >= 4:
                    rows.append((parts[3].strip('"'), parts[2].strip('"'),
                                 os.path.basename(os.path.dirname(p))))
    mal = pd.DataFrame(rows, columns=["user", "date", "scenario"])
    mal["day"] = pd.to_datetime(mal["date"], format=FMT, errors="coerce").dt.normalize()
    mal = mal.dropna(subset=["day"]).drop_duplicates(["user", "day"])
    F = F.merge(mal[["user", "day", "scenario"]].assign(label=1), on=k, how="left")
    F["label"] = F["label"].fillna(0).astype(np.int8)
    F["insider_user"] = F.groupby("user")["label"].transform("max").astype(np.int8)
    F.to_parquet(FEAT, index=False)
    shutil.rmtree(SCRATCH, ignore_errors=True)

F = pd.read_parquet(FEAT)
FEATS = [c for c in F.columns if c not in ("user", "day", "label", "insider_user", "scenario")]
print(f"User-days: {len(F):,}  users: {F.user.nunique():,}  insiders: "
      f"{F.loc[F.insider_user==1,'user'].nunique()}  malicious days: {int(F.label.sum())}  "
      f"features: {len(FEATS)}")

# ------------------------------------------------- 3. user-level split
rng = np.random.RandomState(42)
def split_users(users):
    u = np.array(sorted(users)); rng.shuffle(u); n = len(u)
    return u[: int(.6 * n)], u[int(.6 * n): int(.8 * n)], u[int(.8 * n):]
ins = F.loc[F.insider_user == 1, "user"].unique(); nor = F.loc[F.insider_user == 0, "user"].unique()
S = [np.concatenate(p) for p in zip(split_users(ins), split_users(nor))]
part = {n: F[F.user.isin(u)].reset_index(drop=True) for n, u in zip(["train", "val", "test"], S)}
sc = StandardScaler().fit(part["train"][FEATS])
def XY(df):
    X = np.clip(sc.transform(df[FEATS]).astype(np.float32), -8, 8)
    return torch.tensor(X, device=DEV), torch.tensor(df["label"].values, dtype=torch.float32, device=DEV)
Xtr, ytr = XY(part["train"]); Xva, yva = XY(part["val"]); Xte, yte = XY(part["test"])
yv, yt = yva.cpu().numpy(), yte.cpu().numpy()
tr_users = part["train"]["user"].values
tr_ins = np.array(sorted(part["train"].loc[part["train"].insider_user == 1, "user"].unique()))
for n, d in part.items():
    print(f"  {n:5s}: {len(d):>8,} user-days  {d.user.nunique():>5} users  "
          f"{d.loc[d.insider_user==1,'user'].nunique():>3} insiders  {int(d.label.sum()):>4} malicious days")

def insider_mask(frac, seed):
    """Keep all normal users and a random `frac` of the training insiders."""
    if frac >= 1: return None
    keep = np.random.RandomState(100 + seed).choice(tr_ins, max(2, int(round(frac * len(tr_ins)))), replace=False)
    return torch.tensor(~np.isin(tr_users, tr_ins) | np.isin(tr_users, keep), device=DEV)

# --------------------------------------- 4. GPU statevector simulator (exact)
# Wire 0 = most-significant qubit (PennyLane convention).
NQ = CFG["n_qubits"]
def _apply_1q(st, U, w):
    st = st.movedim(w + 1, -1)
    st = torch.einsum("b...j,bij->b...i", st, U) if U.dim() == 3 else st @ U.T
    return st.movedim(-1, w + 1)
def _cnot(st, c, t):
    s = st.clone(); idx = [slice(None)] * st.dim(); idx[c + 1] = 1
    tt = t if t < c else t - 1; s[tuple(idx)] = st[tuple(idx)].flip(tt + 1); return s
def _ry(th):
    c, s = torch.cos(th / 2), torch.sin(th / 2)
    return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(torch.complex64)
def _rot(phi, th, om):  # PennyLane Rot = RZ(om) RY(th) RZ(phi)
    c, s = torch.cos(th / 2), torch.sin(th / 2)
    e = lambda a: torch.exp(1j * a.to(torch.complex64) / 2)
    return torch.stack([torch.stack([e(-(phi + om)) * c, -e(phi - om) * s]),
                        torch.stack([e(-(phi - om)) * s, e(phi + om) * c])])
def circuit_reup(x, W, A):
    """Data re-uploading circuit. For each layer l: RY(A[l,w]*x_w) on every wire,
    then a strongly-entangling layer (Rot + CNOT ring, range (l mod (n-1))+1).
    x (B,NQ), W (L,NQ,3), A (L,NQ). Returns <Z_w> for each wire, shape (B,NQ)."""
    B = x.shape[0]
    st = torch.zeros((B,) + (2,) * NQ, dtype=torch.complex64, device=x.device)
    st[(slice(None),) + (0,) * NQ] = 1
    for l in range(W.shape[0]):
        for w in range(NQ): st = _apply_1q(st, _ry(A[l, w] * x[:, w]), w)
        for w in range(NQ): st = _apply_1q(st, _rot(W[l, w, 0], W[l, w, 1], W[l, w, 2]), w)
        r = (l % (NQ - 1)) + 1
        for w in range(NQ): st = _cnot(st, w, (w + r) % NQ)
    p = st.abs().pow(2); out = []
    for w in range(NQ):
        pw = p.movedim(w + 1, 1).reshape(B, 2, -1).sum(-1); out.append(pw[:, 0] - pw[:, 1])
    return torch.stack(out, 1)

def crosscheck():
    import pennylane as qml
    dev = qml.device("default.qubit", wires=NQ); L = CFG["q_layers"]
    @qml.qnode(dev, interface="torch")
    def ref(x, W, A):
        for l in range(L):
            qml.AngleEmbedding(A[l] * x, wires=range(NQ), rotation="Y")
            qml.StronglyEntanglingLayers(W[l:l + 1], wires=range(NQ), ranges=[(l % (NQ - 1)) + 1])
        return [qml.expval(qml.PauliZ(i)) for i in range(NQ)]
    g = torch.Generator().manual_seed(7)
    x = torch.rand(5, NQ, generator=g) * math.pi
    W = torch.rand(L, NQ, 3, generator=g) * 2 * math.pi; A = torch.rand(L, NQ, generator=g) * 2
    mine = circuit_reup(x, W, A)
    theirs = torch.stack([torch.stack([t.float() for t in ref(x[i], W, A)]) for i in range(5)])
    err = (mine - theirs).abs().max().item()
    print(f"Statevector simulator vs PennyLane default.qubit: max |diff| = {err:.2e}"); assert err < 1e-4
crosscheck()

# ----------------------------------------------------------------- 5. models
class QLayer(nn.Module):
    def __init__(s):
        super().__init__(); L = CFG["q_layers"]
        s.W = nn.Parameter(torch.rand(L, NQ, 3) * 2 * math.pi); s.A = nn.Parameter(torch.ones(L, NQ))
    def forward(s, x): return circuit_reup(x, s.W, s.A)
class Scale(nn.Module):
    def __init__(s, n): super().__init__(); s.a = nn.Parameter(torch.ones(n))
    def forward(s, x): return x * s.a
class Net(nn.Module):
    """Encoder (d -> h -> 4, tanh) -> middle block -> linear head.
    Middle block: 4-qubit, 3-layer re-uploading circuit (36 rotation + 12 input-scale
    = 48 params) or a classical block with exactly 48 params
    (2 x [Linear(4,4) + tanh + learnable per-unit scale])."""
    def __init__(s, d, quantum, h):
        super().__init__(); s.quantum = quantum
        s.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h, NQ), nn.Tanh())
        s.mid = QLayer() if quantum else nn.Sequential(nn.Linear(NQ, NQ), nn.Tanh(), Scale(NQ),
                                                       nn.Linear(NQ, NQ), nn.Tanh(), Scale(NQ))
        s.head = nn.Linear(NQ, 1)
    def forward(s, x):
        z = s.enc(x)
        return s.head(s.mid(z * math.pi / 2 + math.pi / 2) if s.quantum else s.mid(z)).squeeze(-1)

@torch.no_grad()
def predict(m, X):
    m.eval(); return torch.cat([torch.sigmoid(m(X[i:i + 65536])) for i in range(0, len(X), 65536)]).cpu().numpy()

def fit(quantum, seed, lr, h, mask=None, epochs=None):
    X, y = (Xtr, ytr) if mask is None else (Xtr[mask], ytr[mask])
    epochs = epochs or CFG["epochs"]
    torch.manual_seed(seed); np.random.seed(seed)
    m = Net(X.shape[1], quantum, h).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    pw = torch.tensor((len(y) - y.sum().item()) / max(y.sum().item(), 1), device=DEV)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, state, bad, t0 = -1, None, 0, time.time()
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(X), device=DEV)
        for i in range(0, len(perm), CFG["batch"]):
            b = perm[i:i + CFG["batch"]]; opt.zero_grad(); lossf(m(X[b]), y[b]).backward(); opt.step()
        sch.step()
        ap = average_precision_score(yv, predict(m, Xva))
        if ap > best: best, state, bad = ap, {k: v.detach().clone() for k, v in m.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= CFG["patience"]: break
    m.load_state_dict(state)
    return m, best, time.time() - t0, sum(p.numel() for p in m.parameters())

def evaluate(m):
    pv, pt = predict(m, Xva), predict(m, Xte)
    fpr, tpr, th = roc_curve(yv, pv); thr1 = th[max(np.searchsorted(fpr, CFG["fpr_target"], "right") - 1, 0)]
    ths = np.quantile(pv, np.linspace(0.5, 0.9999, 400))
    thrF = ths[np.argmax([f1_score(yv, pv >= t, zero_division=0) for t in ths])]
    d = part["test"].assign(p=pt); det, delay = [], []
    for _, g in d[d.insider_user == 1].groupby("user"):
        st = g.loc[g.label == 1, "day"].min(); hit = g[(g.day >= st) & (g.p >= thr1)]
        det.append(len(hit) > 0); delay.append((hit.day.min() - st).days if len(hit) else np.nan)
    return dict(pr_auc=average_precision_score(yt, pt), auc=roc_auc_score(yt, pt),
                recall_at_1fpr=float(((pt >= thr1) & (yt == 1)).sum() / yt.sum()),
                f1=f1_score(yt, pt >= thrF, zero_division=0), detected=float(np.mean(det)),
                mttd_days=float(np.nanmedian(delay)) if np.isfinite(delay).any() else np.nan)

# ------------------------------------------------------------ 6. experiments
# P = encoder width sweep on all labels; L = label scarcity at full width.
# h=32 with all labels is shared by both (P "h=32" == L "100%").
SETTINGS = ([("P", f"h={h}", h, 1.0) for h in CFG["widths"]] +
            [("L", f"{int(f*100)}%", CFG["full_width"], f) for f in CFG["insider_fracs"]])
MODELS = [("Hybrid", True), ("Classical", False)]
T0 = time.time(); el = lambda: f"{(time.time()-T0)/60:5.1f} min"

# 6a. learning-rate tuning PER SETTING and PER MODEL (validation PR-AUC only)
print(f"\n[Tuning] lr grid {CFG['lr_grid']} | tuning seed {CFG['tune_seed']} | {CFG['tune_epochs']} epochs")
tune_rows, best_lr = [], {}
for exp, setting, h, frac in SETTINGS:
    for name, q in MODELS:
        sc_ = {}
        for lr in CFG["lr_grid"]:
            _, vap, secs, _ = fit(q, CFG["tune_seed"], lr, h, insider_mask(frac, CFG["tune_seed"]), CFG["tune_epochs"])
            sc_[lr] = vap; tune_rows.append(dict(exp=exp, setting=setting, model=name, lr=lr, val_pr_auc=vap))
        best_lr[(setting, name)] = max(sc_, key=sc_.get)
        edge = " (EDGE OF GRID)" if best_lr[(setting, name)] in (CFG["lr_grid"][0], CFG["lr_grid"][-1]) else ""
        print(f"  {el()} [{exp}] {setting:>5} {name:9s} val PR-AUC "
              + " ".join(f"{k:.0e}:{v:.3f}" for k, v in sc_.items()) + f" -> {best_lr[(setting, name)]:.0e}{edge}")
pd.DataFrame(tune_rows).to_csv(f"{OUT}/tuning.csv", index=False)

# 6b. final runs: 5 fresh seeds, tuned lr, 40 epochs, test set evaluated once
print(f"\n[Final] seeds {CFG['seeds']} | {CFG['epochs']} epochs | early stopping on validation PR-AUC")
rows = []
for exp, setting, h, frac in SETTINGS:
    for seed in CFG["seeds"]:
        for name, q in MODELS:
            lr = best_lr[(setting, name)]
            m, vap, secs, npar = fit(q, seed, lr, h, insider_mask(frac, seed))
            r = dict(exp=exp, setting=setting, width=h, insider_frac=frac, model=name, seed=seed, lr=lr,
                     params=npar, val_pr_auc=vap, train_s=secs, **evaluate(m))
            rows.append(r); pd.DataFrame(rows).to_csv(f"{OUT}/all_runs.csv", index=False)
            print(f"  {el()} [{exp}] {setting:>5} {name:9s} seed {seed} lr {lr:.0e}: PR-AUC {r['pr_auc']:.3f} "
                  f"AUC {r['auc']:.3f} R@1% {r['recall_at_1fpr']:.3f} F1 {r['f1']:.3f} "
                  f"det {r['detected']:.0%} ({npar} p, {secs:.0f}s)")
R = pd.DataFrame(rows)
R = pd.concat([R, R[(R.exp == "P") & (R.setting == f"h={CFG['full_width']}")].assign(exp="L", setting="100%")],
              ignore_index=True)
R.to_csv(f"{OUT}/all_runs.csv", index=False)

# ------------------------------------------------------------ 7. statistics
mets = ["pr_auc", "auc", "recall_at_1fpr", "f1", "detected", "mttd_days"]
R["collapsed"] = (R.pr_auc < CFG["collapse_pr_auc"]).astype(float)
summ = R.groupby(["exp", "setting", "model"], sort=False)[mets + ["collapsed", "params", "lr"]].agg(["mean", "std"])
summ.to_csv(f"{OUT}/summary.csv")

print("\n========== HYBRID minus CLASSICAL, paired by seed (test set) ==========")
print("   mean delta (seeds hybrid won / n, one-sided Wilcoxon p)")
V = []
for (exp, setting), g in R.groupby(["exp", "setting"], sort=False):
    H = g[g.model == "Hybrid"].set_index("seed"); C = g[g.model == "Classical"].set_index("seed")
    out = [f"[{exp}] {setting:>5}"]
    for mt in mets[:-1]:
        dlt = (H[mt] - C[mt]).dropna(); wins = int((dlt > 0).sum())
        try: p = wilcoxon(dlt, alternative="greater").pvalue if (dlt != 0).any() else 1.0
        except ValueError: p = 1.0
        V.append(dict(exp=exp, setting=setting, metric=mt, mean_delta=dlt.mean(), wins=wins, n=len(dlt), p_one_sided=p))
        out.append(f"{mt} {dlt.mean():+.3f} ({wins}/{len(dlt)}, p={p:.3f})")
    out.append(f"collapse H {H.collapsed.mean():.0%} / C {C.collapsed.mean():.0%}")
    print("  ".join(out))
V = pd.DataFrame(V); V.to_csv(f"{OUT}/paired_deltas.csv", index=False)

sig = V[(V.mean_delta > 0) & (V.p_one_sided < 0.05)]
print("\nHYBRID ADVANTAGES THAT HOLD (hybrid better, one-sided Wilcoxon p < 0.05 over 5 paired seeds):")
print(sig.to_string(index=False) if len(sig) else "  none")
neg = V[(V.mean_delta < 0) & (V.wins == 0)]
print("\nSettings where the CLASSICAL model won every seed (report these too):")
print(neg.to_string(index=False) if len(neg) else "  none")

# ------------------------------------------------- 8. LaTeX rows and figure
def pm(a, b, f="{:.3f}"): return f"{f.format(a)} $\\pm$ {f.format(0 if np.isnan(b) else b)}"
def tex_rows(exp, label_col):
    out = []
    for (e, setting, model), r in summ.loc[[exp]].iterrows():
        out.append(f"{setting.replace('%', chr(92) + '%')} & {model} & {int(r[('params', 'mean')])} & "
                   + " & ".join(pm(r[(m_, 'mean')], r[(m_, 'std')]) for m_ in ["pr_auc", "auc", "recall_at_1fpr", "f1"])
                   + f" & {r[('detected', 'mean')]*100:.0f}\\% & {r[('collapsed', 'mean')]*100:.0f}\\% \\\\")
    return out
open(f"{OUT}/table_width_rows.tex", "w").write("\n".join(tex_rows("P", "Width")) + "\n")
open(f"{OUT}/table_labels_rows.tex", "w").write("\n".join(tex_rows("L", "Insiders")) + "\n")
print(f"\nLaTeX rows: {OUT}/table_width_rows.tex, table_labels_rows.tex")

fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.7))
for name, fmt in [("Hybrid", "-o"), ("Classical", "--s")]:
    g = R[(R.exp == "P") & (R.model == name)].groupby("params")["pr_auc"]
    ax[0].errorbar(g.mean().index, g.mean().values, yerr=g.std().values, fmt=fmt, color="k", ms=3, lw=1, capsize=2, label=name)
    g = R[(R.exp == "L") & (R.model == name)].groupby("insider_frac")["pr_auc"]
    ax[1].errorbar(np.array(g.mean().index) * 100, g.mean().values, yerr=g.std().values, fmt=fmt, color="k", ms=3, lw=1, capsize=2, label=name)
ax[0].set_xscale("log"); ax[0].set_xlabel("Trainable parameters"); ax[0].set_ylabel("Test PR-AUC")
ax[1].set_xscale("log"); ax[1].set_xticks([10, 25, 50, 100]); ax[1].set_xticklabels(["10", "25", "50", "100"])
ax[1].set_xlabel("Training insiders kept (%)"); ax[1].set_ylabel("Test PR-AUC")
for a in ax: a.legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT}/pr_auc_width_labels.pdf"); plt.savefig(f"{OUT}/pr_auc_width_labels.png", dpi=300)
json.dump({**CFG, "best_lr": {f"{k[0]}|{k[1]}": v for k, v in best_lr.items()}}, open(f"{OUT}/config.json", "w"), indent=2)
print(f"Done in {el()}. All outputs in {OUT}")


$ pip -q install pennylane pyarrow
$ apt-get -qq install -y lbzip2 > /dev/null
Mounted at /content/drive
Device: cuda Tesla T4
User-days: 330,452  users: 1,000  insiders: 72  malicious days: 986  features: 35
  train:  196,283 user-days    599 users   43 insiders   571 malicious days
  val  :   66,720 user-days    200 users   14 insiders   190 malicious days
  test :   67,449 user-days    201 users   15 insiders   225 malicious days
Statevector simulator vs PennyLane default.qubit: max |diff| = 2.09e-07

[Tuning] lr grid [0.0003, 0.001, 0.003, 0.01, 0.03] | tuning seed 0 | 20 epochs
    3.4 min [P]   h=4 Hybrid    val PR-AUC 3e-04:0.031 1e-03:0.042 3e-03:0.102 1e-02:0.079 3e-02:0.078 -> 3e-03
    3.6 min [P]   h=4 Classical val PR-AUC 3e-04:0.014 1e-03:0.060 3e-03:0.071 1e-02:0.057 3e-02:0.040 -> 3e-03
    6.4 min [P]   h=8 Hybrid    val PR-AUC 3e-04:0.021 1e-03:0.069 3e-03:0.087 1e-02:0.095 3e-02:0.079 -> 1e-02
    6.5 min [P]   h=8 Classical val PR-AUC 3e-04:0.030 1e-03:0.032 3e-03:0